# 01 — Mask-to-Box Inspection

**Purpose:** inspect the pre-registered `mask_to_box` conversion on RescueNet / FloodNet+ masks **before** any experimental results are collected. Visualize:

1. Raw semantic masks.
2. Connected-component boxes **before** filtering.
3. Boxes **after** filtering (min area 32 px², max < 50% image, aspect 1:10–10:1, stuff classes excluded).
4. Final COCO-style JSON sanity (no empty categories, sane bbox stats).

> Any change to the filtering rules requires a `docs/change_log.md` entry — the rules are frozen.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import yaml

from data.mask_to_box.filter import (
    build_class_lookup,
    connected_components_to_boxes,
    filter_boxes,
    masks_to_coco,
)

with open("configs/datasets/rescuenet.yaml") as fh:
    cfg = yaml.safe_load(fh)
lookup = build_class_lookup(cfg)
print(lookup)

In [ ]:
# TODO: load one real mask image (data/raw, outside git)
mask = np.zeros((512, 512), dtype=np.uint8)
mask[100:200, 100:250] = 1   # building
mask[300:450, 300:350] = 2   # road (stuff, excluded)

boxes = connected_components_to_boxes(mask == 1)
kept, m = filter_boxes(boxes, image_h=512, image_w=512)
print("raw boxes:\n", boxes)
print("kept:\n", kept)

fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(mask, cmap="tab10"); ax[0].set_title("mask")
for b in kept:
    x1, y1, x2, y2 = b
    ax[1].add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor="red", lw=2))
ax[1].imshow(mask, cmap="gray"); ax[1].set_title("filtered boxes")
plt.show()

## Checks
- Stuff classes (road/tree/grass/water/sand) never produce annotations.
- Region-level damage classes are flagged `region_level: true` in categories.
- Minimum area, maximum area fraction, and aspect-ratio rules hold on all boxes.